# 自己回帰型ニューラルネットワークと自然言語処理


自己回帰型ニューラルネットワーク（Recurrent Neural Network：RNN）を用いた自然言語処理の記述方法を学習します．  
今回は長短期記憶（Long Short Tem Memory：LSTM）を持ったRNNを使います．

**目標：RNNを用いた自然言語処理の記述を理解**

---

例題では，PyTorchのWord2Vecを使って，回帰問題（文章生成）を解きます．  
演習では，GensimのWord2Vecを使って，分類問題（文章分類）を解きます．  
※[データのフォーマットが異なる](https://drive.google.com/file/d/1hmmDBO_-PFmtuB9XMBUXvEgrPb5_mS5D/view?usp=sharing)ので気をつけましょう．



---
## この教材について

「3分で学ぶPyTorch」シリーズの **NLP（自然言語処理） 基礎編（第1回）** の演習パートです。

このノートブックは**回答ファイル（ans）**です。全演習の解答が入っています。まずは演習ファイル（task）に挑戦してから参照してください。

この教材と関連記事は note で無料公開しています。
シリーズ一覧: https://note.com/technosend/m/m84d841b6d067

---

## 今回使う文章データ
これまでとは異なり，ごく短いデータを用意した．  

- 例題：文章生成での使い方
  - 1単語ずつ入力して文章ごとに，その文章の流れを学習させる
  - 学習データ：`新鮮な魚を焼く`, `珈琲牛乳が美味しい`　の2文  
  - テストデータ：なし
- 演習：文章分類での使い方
  - 1単語ずつ入力して文章ごとに，その文章が肯定的か否定的かを学習させる
    - 肯定的な文をクラス0，否定的な文をクラス1とする


|     分類    |  入力データ  |  教師データ  |
|     ---      |     ----      |      ----    |
|  学習データ  |  `今は焼肉を食べたい`  |  0  |
|             |  `魚は一生食べたくない`  |  1  |
| テストデータ  |  `珈琲牛乳が好きだ`  |  0  |
|             |  `辛いものは苦手だ`  |  1  |

## 前準備

形態素解析を扱うためのライブラリ（[MeCab](https://pypi.org/project/mecab-python3/#description)）とWord2Vecを扱うためのライブラリ（[Gensim](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec)）をColabratoryにインストールし，Word2Vecの学習済みモデルのダウンロード，データセットの前処理で使う関数群を用意する．

---


### 前準備1. MeCabとGensimのインストール

形態素解析を扱うためのライブラリ（MeCab）とWord2Vecを扱うことができるライブラリ（Gensim）をインストール，インポートする．

---


#### 前準備1のコマンド

In [ ]:
!pip install gensim -q
!pip install mecab-python3 -q  # バージョン固定解除 (0.996.6rc2は古い)
# mecab-python3 1.0以降はシステムMeCabのインストール不要
from gensim.models.word2vec import Word2Vec
import MeCab
from IPython.display import clear_output

### 前準備2. Word2Vecの学習済みモデルのダウンロードと読み込み

Word2Vecの学習済みモデルである白ヤギモデルをダウンロードし読み込む．

#### 前準備2のコード

In [ ]:
# 白ヤギモデルのダウンロードと解凍
# 下記URLは白ヤギコーポレーション(2017年公開)のS3バケットです。
# アクセス不可の場合は代替モデル(chiVe等)の使用を検討してください。
# 参考: https://github.com/WorksApplications/chiVe
!wget "http://public.shiroyagi.s3.amazonaws.com/latest-ja-word2vec-gensim-model.zip"
!unzip "latest-ja-word2vec-gensim-model.zip"
clear_output()

# Word2Vecモデルの読み込み
model = Word2Vec.load("word2vec.gensim.model")
wv = model.wv

# モデルの重みを取得
weight = wv.vectors
print("Word2Vecの重みの次元数(語彙数, ベクトルの次元数) :", weight.shape)

# 語彙数を取得
vocab_size = len(wv.key_to_index)  # gensim 4.x: vocab → key_to_index
print("語彙数 :", vocab_size)

# ベクトルの次元数を取得
embedding_size = wv.vector_size
print("ベクトルの次元数 :", embedding_size)

# IDから逆引きを行うためのlist形式の配列（テスト時に使用）を取得
index2word_list = list(wv.index_to_key)  # gensim 4.x: index2word → index_to_key

  - 白ヤギモデルをダウンロードする
    - ダウンロード元：白ヤギコーポレーションのブログ[word2vecの学習済み日本語モデルを公開します](https://aial.shiroyagi.co.jp/2017/02/japanese-word2vec-model-builder/) より

  - ダウンロードした白ヤギモデルを読み込み，今回の例題・演習で使用するパラメータを取得  
    - wv：白ヤギモデルを読み込んだWord2Vec
    - weight：白ヤギモデルの重み
    - vocab_size：白ヤギモデルの語彙数
    - embedding_size：単語をベクトルへ変換したときのベクトルの次元数
    - index2word_list：IDから逆引きを行うためのlist形式の配列

### 前準備3. functionsのダウンロードとインポート

文章の前処理で使う関数が格納されたモジュールfunctionsをダウンロードし，インポートする．  

※functionsはこの講義専用に作成したので，[PyPi](https://pypi.org/)などのパッケージ管理ツールで公開されているものではないので注意．

---


#### 前準備3のコード

In [ ]:
!pip install gdown --upgrade -q  # gdown 4.6以降はアップグレード推奨
import gdown
# 旧形式URL変換済み: uc?export=download は gdown 4.6以降で不安定
file_id = "182oAY1TFgcer5MLAqk5tBK0e7HoCI7Um"  # Google Drive ファイルID
gdown.download(id=file_id, output="functions.py", quiet=True)
clear_output()

import functions

print("ベクトル化のプロセスを試してみる")
# 1. データの用意
text_list = ["ここは夜景が綺麗です", "朝日が眩しい"]

# 2. 各文章を分かち書き
text_list = functions.word_tokenize(text_list)
print("分かち書きした結果", text_list)

# 3. 単語列を単語IDに変換
text_list = functions.text_to_id_list(text_list, index2word_list)
print("単語IDに変換した結果", text_list)

##### functionsを使った前処理の方法

前処理としては，文章の分かち書きや単語IDへの変換を行う．


  1. データの用意
    - 処理する文章をlist形式で宣言  
    ```python
    text_list = ["ここは夜景が綺麗です", "朝日が眩しい"]
    ```

  2. 各文章を分かち書き
    - 各文章に対して分かち書きを行い，単語間がスペースで区切られた文章（単語列）を作成
    ```python
    text_list = functions.word_tokenize(text_list)
    # 第１引数：分かち書きを行う文章(list)
    # 戻り値1：分かち書きした結果（list）
    ```

  3. 単語列を単語ID列に変換
    - 単語列の各単語を単語IDへ変換
    ```python
    text_list = functions.text_to_id_list(text_list, index2word_list)
    # 第１引数：変換する単語列(list)
    # 第２引数：単語が格納された配列(list)
    # 戻り値1：コーパスに従って取得した単語ID(list)
    ```

## 例題 文章生成

**Embedding層・LSTM1層のRNNの作成**  
文章を単語に分解し時系列データとして順次入力し，次の単語を予測する回帰問題を解く．

 

### 例題1. ライブラリのインポート

深層学習演算ライブラリPyTorchなどのライブラリ，パッケージ，モジュールをインポートする．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=14XdT7XWs6JzTil6JhZZw7aM3VpAxdYH2&sz=w400">




#### 例題1のコード

In [ ]:
# 例題1. ライブラリのインポート

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

#### 今回使うパッケージ，ライブラリ，モジュール一覧

- [torch](https://pytorch.org/docs/stable/torch.html)：多次元テンソルのデータ構造とそのテンソルのための算術演算が組み込まれたパッケージ
- [torch.nn](https://pytorch.org/docs/stable/nn.html)：ニューラルネットワークを定義するためのパッケージ  
nnという略称を与えることが多い
- [torch.optim](https://pytorch.org/docs/stable/optim.html)：最適化器を宣言するためのパッケージ  
optimという略称を与えることが多い
- [numpy](https://numpy.org/doc/stable/user/whatisnumpy.html)：行列演算を行うライブラリ（今回は画像の表示のために使用）  
npという略称を与えることが多い

<font color="blue">【TASK】</font>パッケージをインポートしましょう

前処理でインポート済みのもの

- **<font color="red">【NEW!】</font>**[MeCab](https://pypi.org/project/mecab-python3/#description)：形態素解析を行うライブラリ（白ヤギモデルに合わせたバージョンを指定）
- **<font color="red">【NEW!】</font>**[gensim](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec)：Word2Vecを扱うためのライブラリ
- functions：前準備3でダウンロード，インポートしたもの  
  文章を分かち書きしたりWord2Vecに変換したりする関数が格納されたモジュール  
  ※この講義専用のモジュールで，PyPiなどのパッケージ管理ツールで公開されているものではないので注意


### 例題2. ニューラルネットワークの定義

ニューラルネットワーククラスを定義して，そのクラスのインスタンスを宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1jl1W_RW7HrU0ksk1a0XrSq6CyldXF4qZ&sz=w400">

#### 例題2のコード

In [ ]:
# 例題2. ニューラルネットワークの定義

# 1. ニューラルネットワーククラスの定義
class TextGenerator(nn.Module):
    def __init__(self, embedding_size, weight):
        super(TextGenerator, self).__init__()
        self.embed = nn.Embedding.from_pretrained(weight) 
        self.lstm1 = nn.LSTM(embedding_size, embedding_size, num_layers=1)

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.lstm1(x)
        return x
 
    # Word2Vecで単語IDをベクトルに変換
    def encode(self, x):
        return self.embed(x)

# 2. インスタンスの宣言
text_generator = TextGenerator(embedding_size, torch.tensor(weight))
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
text_generator.to(device)

#### 1. ニューラルネットワーククラスの定義  
  - nn.Moduleを継承したクラス「TextGenerator」を定義
    - コンストラクタの引数にembedding_size, weightを持つ
      - embedding_size：単語からベクトルへ変換したときのベクトルの次元数
      - weight：学習済みWord2Vecモデルの重み
  - \_\_init\_\_()，forward()，encode()を定義
    ```python
    class TextGenerator(nn.Module):
        def __init__(self, embedding_size, weight):
            super(TextGenerator, self).__init__()
            # 【TASK】Embedding層とLSTM層を宣言
        def forward(self, x):
            # 【TASK】順伝播のパスを定義
            return x
        def encode(self, x):
            return self.embed(x)
    ```

  - \_\_init\_\_()では，Embedding層とLSTM層を宣言
    - [super()](https://docs.python.org/ja/3/library/functions.html#super)を呼び出す
    - 一つの層につき一つ，nnパッケージ内のクラスのインスタンスを宣言するので，今回は二つ宣言 
    - **<font color="red">【NEW!】</font>**Embedding層は[nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)クラスを用いて宣言
      - Word2Vecにより単語IDをベクトルに変換
      - from_pretrained()を使うことで事前に学習済みモデルの重みを与える
      - from_pretrained()で宣言した重みはこのニューラルネットワークの学習の対象外
      ```python
      self.embed = nn.Embedding.from_pretrained(weight)
      # 第１引数：学習済みの重み（torch.tensor）
      ```
    - LSTM層は[nn.LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)クラスを用いて宣言
    ```python
    self.lstm1 = nn.LSTM(embedding_size, embedding_size, num_layers=1)
    # 第１引数：入力次元数（int）
    # 第２引数：出力次元数（int）
    # 第３引数（オプション）：レイヤー数（int）， デフォルト1
    ```


  <font color="blue">【TASK】</font>Embedding層とLSTM層をnn.Embedding.from_pretrained()とnn.LSTMクラスを用いて宣言しましょう  
  ニューラルネットワークの構成は下記の通りです  
  - Embedding層：入力vocab_size，出力embedding_size  
    - ここでは，from_pretrained()の引数に学習済みの重みを渡す  
        これにより重みの次元数によって入出力サイズが決定  
  - LSTM層1：入力embedding_size，出力  embedding_size，レイヤー数1

  - forward()では，順伝播のパスを定義  
    - forward()は外部からの入力データxを受け取り，順伝播を行う
    - <font color="red">【NEW!】</font>nn.Embeddingクラスのインスタンスに()をつけて引数を与えて呼び出すと，クラスメソッドである\_\_call_\_()が呼び出され，引数（整数値）をベクトルに変換したものを出力  
    - nn.LSTMクラスのインスタンスに()をつけて引数を与えて呼び出すと，クラスメソッドである\_\_call_\_()が呼び出され，引数に対するLSTM層の計算結果が返される  
  ※nn.LSTMクラスの\_\_call_\_()の戻り値はLSTMの出力とセル状態の二つあるが，今回はLSTMの出力のみ取得
    ```python
    x = self.embed(x) 
    x, _ = self.lstm1(x) 
    ```

  <font color="blue">【TASK】</font>順伝播のパスを定義しましょう  
  ある時刻の単語（整数値）を入力すると想定したとき順伝播のパスの構成は下記の通りです
    1. Embedding層  
    2. LSTM層1  

  - encode()では，Word2Vecで単語IDをベクトルに変換するパスを定義
    - 学習済みWord2Vecモデルの重みをセットしたEmbedding層（embed）を用いて単語IDをベクトルに変換  
    ```python
    def encode(self, x):
        return self.embed(x)
    ```


#### 2. インスタンスの宣言  

- text_generatorという名前でインスタンスを宣言
```python
text_generator = #【TASK】ニューラルネットワーククラスのインスタンスの宣言
```

- GPUにセットアップ
```python
device = #【TASK】GPUの指定
#【TASK】GPUにセットアップ
```



  - 今回はWord2Vecのモデルや学習する文章によってニューラルネットワークの構成が変わるので，引数でパラメータを渡す
  ```python
  text_generator = TextGenerator(embedding_size, torch.tensor(weight))
  # 第1引数：学習済みWord2Vecモデルが出力するベクトルの次元数(int)
  # 第2引数：学習済みWord2Vecモデルの重み(tensor)
  ```

- ニューラルネットワーククラスのインスタンスの宣言

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスを宣言しましょう 



- GPUの指定
  - デバイス指定用の変数deviceを宣言しましょう
  - [torch.device](https://pytorch.org/docs/stable/tensor_attributes.html#torch.torch.device)クラスを使って変数deviceを初期化しましょう
  - torch.deviceクラス宣言時に"cuda:0"を与えましょう

  ```python
  device = torch.device("cuda:0")
  ```

　　<font color="blue">【TASK】</font>GPUを指定しましょう

- GPUにセットアップ
  - Moduleクラスにもto()がある
  - Tensor型変数同様にto()を使ってGPUにデータを渡すことができる
  - 学習時にGPUを使う場合は，ニューラルネットワーククラスのインスタンスをGPUに渡す

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスをGPUにセットアップしましょう  
  - [to()](https://pytorch.org/docs/1.9.1/generated/torch.Tensor.to.html)を使ってGPUにセットアップしましょう
  ```python
  text_generator.to(device)
  ```


### 例題3. 誤差関数・最適化器の設定

誤差関数と最適化器を宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1oS8_oitSvcQ9f5_uYMrDXyC3P_vrM7jp&sz=w400">



#### 例題3のコード

In [ ]:
# 例題3. 誤差関数・最適化器の設定

# 1.誤差関数の宣言
criterion = nn.MSELoss()

# 2.最適化器の宣言
optimizer_text_generator = optim.Adam(text_generator.parameters())

#### 1. 誤差関数の宣言  
  - 平均二乗誤差（回帰問題のため）を計算するcriterionを宣言

    ```python
    criterion = # 【TASK】誤差関数の宣言
    ```



  - 平均二乗誤差は[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)クラスを用いて宣言
  - nn.MSELoss()で宣言したインスタンスは，二つの引数の平均二乗誤差を返す  

  ```python
  criterion = nn.MSELoss()
  ```
  
<font color="blue">【TASK】</font>誤差関数を宣言しましょう
- 平均二乗誤差関数を使いましょう
- nn.MSELossクラスを用いて宣言しましょう

#### 2. 最適化器の宣言

  - Adamを計算するoptimizer_text_generatorを宣言

    ```python
    optimizer_text_generator = # 【TASK】最適化器の宣言
    ```

  - Adamはoptim.Adamクラスを用いて宣言
  ```python
  optimizer_text_generator = optim.Adam(text_generator.parameters())
  # 第１引数：ニューラルネットワークのパラメータ
  ```

<font color="blue">【TASK】</font>最適化器を宣言しましょう  
- Adamを使いましょう
- [optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#adam)クラスを用いて宣言しましょう  
- 引数の構成は下記の通りです  
  - ニューラルネットワークのパラメータtext_generatorのパラメータ

### 例題4. データセットの準備

自作した関数make_dataset()を使ってデータローダーを作成する．


---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=17fS-oMI83rSL7SxN_GyKHjxP-FO-R3aQ&sz=w400">


#### 例題4のコード

In [ ]:
# 例題4. データセットの準備

# 1. データセットを作成する関数の定義
def make_dataset(id_list):
    # 学習データをTensor型に変換
    id_list = torch.tensor(id_list)

    # 入力データと教師データに分割
    # 時刻 0 ~ (T-1) を入力データ
    # 時刻 1 ~ T を教師データ
    inputs = id_list[:, :-1].permute(1,0)
    targets = id_list[:, 1:].permute(1,0)

    print("\nデータセットの概要")
    print("入力データの次元数 :", inputs.shape)
    print("入力データの中身（単語ID列）\n", inputs)
    print("教師データの次元数 :", targets.shape)
    print("教師データの中身（単語ID列）\n", targets)

    # 入力データと教師データをひとつにまとめる
    train_loader = [(inputs, targets)]

    return train_loader

# 2. 文章の用意
text_list = ["新鮮な魚を焼く", "珈琲牛乳が美味しい"]

# 3. 各文章を分かち書き
text_list = functions.word_tokenize(text_list)

# 4. 単語IDに変換
text_list = functions.text_to_id_list(text_list, index2word_list)

# 5. データセットの作成
train_loader = make_dataset(text_list)
seq_len = len(train_loader[0][0])

#### 1. データセットを作成する関数の定義
  - make_dataset()という名前でデータセットを作成する関数を定義
    - 学習データの変数を受け取る
    - 学習データのデータローダーを返す

  ```python
  train_loader = make_dataset(id_list) 
  # 第1引数：学習データの変数，単語IDに変換した文章(list)
  # 戻り値1：データローダー(list)
  ```

- 学習データをTensor型に変換
```python
id_list = torch.tensor(id_list)
```
- 入力データと教師データに分割  
  - データ長（文章の長さ）をTとして下記の通りにスライス  
    - 入力データ：時刻 0 ~ (T - 1)
    - 教師データ：時刻 1 ~ T
  - データ長，データの順番になるように並べ替え
  
  ```python
  inputs = id_list[:, :-1].permute(1,0)
  targets = id_list[:, 1:].permute(1,0)
  ```
- 入力データと教師データをひとつにまとめる  
  - 入力データと教師データをtuple形式でまとめ，さらにlist形式でまとめ直す
```python
train_loader = [(inputs, targets)]
```

#### 2. 文章の用意
  - text_listという名前で文章をlist形式で宣言

  ```python
  text_list = # 文章を宣言
  ```

  - 用意する文章は`新鮮な魚を焼く`，`珈琲牛乳が美味しい`とする
  ```python
  text_list = ["新鮮な魚を焼く", "珈琲牛乳が美味しい"]
  ```

#### 3. 各文章を分かち書き
  
  - text_listという名前で文章を分かち書きした単語列を格納する変数を宣言

  ```python
  text_list = # 文章を分かち書き
  ```


  - 分かち書きにはfunctions.word_tokenize()を使う  
  ```python
  text_list = functions.word_tokenize(text_list)
  # 第1引数：分かち書きする文章(list)
  # 戻り値1：分かち書きした結果の文章データ(list)
  ```

#### 4. 分かち書きされた文章を単語IDに変換
  - text_listに単語IDに変換したlist形式の配列を再代入


  - 単語IDへの変換は単語が格納された配列を使う
  
  ```python
  text_list = functions.text_to_id_list(text_list, index2word_list)
  # 第１引数：変換する単語列(list)
  # 第２引数：単語が格納された配列(list)
  # 戻り値1：コーパスに従って取得した単語ID(list)
  ```

#### 5. データセットの作成

- train_loaderという名前で学習データのデータローダー用の変数を宣言  
- make_dataset()を用いてデータセットを作成

```python
train_loader = # データセットの作成
```



### 例題5. 学習

教師データとの誤差を計算し，パラメータを更新する．  

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1A6TjumoBejWpKGvD6TDQeivN1tpeEBp4&sz=w400">

#### 例題5のコード：前半の学習部分

In [ ]:
# 例題5. 学習

# 1. 学習ループの作成
epochs = 1000

# エポックのループ
for epoch in range(epochs):
    # 学習データのデータローダーのループ
    for data in train_loader:

        # 2. ニューラルネットワークへのデータの入力
        # 文章とターゲットに分割
        inputs, targets = data
        inputs = inputs.to(device)
        targets = targets.to(device)
        outputs = text_generator(inputs)
        
        # 3. 誤差の計算（BPTTパターン1：全ての時刻に教師データがある場合）
        loss = torch.tensor(0.0).to(device)
        for i in range(seq_len - 1):
            loss += criterion(outputs[i], text_generator.encode(targets[i]))

        # 4. 誤差逆伝播とパラメータの更新
        optimizer_text_generator.zero_grad()
        loss.backward()
        optimizer_text_generator.step()
      
    # 現在の誤差の値の表示
    if (epoch + 1) % 100 == 0:
        print("epoch :", epoch + 1, "学習誤差 :", loss.item())

#### 1. 学習ループの作成  
- ミニバッチ学習を行う学習ループの作成
- epochsという名前でエポック用の変数を宣言
- 外側にエポック，内側に学習データのデータローダーのループを作成
```python
epochs = # 【TASK】エポック数
for epoch in # 【TASK】エポック
      for data in # 【TASK】学習データのデータローダー
```

<font color="blue">【TASK】</font>学習ループを作成しましょう  
ループの設定は下記の通りです  
  - エポック
    - エポック数1000
    - [range](https://docs.python.org/ja/3/library/stdtypes.html#range)クラスを使ってループさせましょう
  - 学習データのデータローダー
    - ループ対象：学習データのデータローダーtrain_loader
    - [for](https://docs.python.org/ja/3/reference/compound_stmts.html#for)を使ってtrain_loaderをループさせましょう

#### 2. ニューラルネットワークへのデータの入力  

- GPUにセットアップ
- outputsという名前で出力用の変数を宣言
```python
# 文章とターゲットに分割
inputs, targets = data
inputs = # 【TASK】GPUにセットアップ
targets = # 【TASK】GPUにセットアップ
outputs = # 【TASK】ニューラルネットワークからの出力
```


  
<font color="blue">【TASK】</font>ニューラルネットワークへデータを入力しましょう
  - 入力inputsをGPUにセットアップしましょう
  - 教師データtargetsをGPUにセットアップしましょう
  - inputsをニューラルネットワークに与えて出力outputsを取得しましょう

#### 3. 誤差の計算（BPTTパターン1：全ての時刻に教師データがある場合）

- loss という名前で誤差計算の結果用の変数を宣言 
- データ長だけ繰り返すループをfor文を使って作成

  ```python
  loss = # 【TASK】0に初期化
  for i in # 【TASK】データ長:
      loss += # 【TASK】誤差計算
  ```



  - lossという変数をTensor型の変数として宣言し，0.0で初期化
  - GPUにセットアップ
  ```python
  loss = torch.tensor(0.0).to(device)
  ```

- 誤差計算はnn.MSELossクラスやnn.CrossEntropyLossクラスなどの誤差関数クラスのインスタンスに引数を二つ与えて行う
```python
loss = criterion(outputs, targets)
# 第1引数：ニューラルネットワークの出力
# 第2引数：教師データ
```

  - for文でループさせながら各時刻ごとの誤差を計算しlossに足す  
  ループ回数はデータ長（今回はseq_len - 1）だけ繰り返す
    - -1するのはデータセットtrain_dataは用意した文章の長さから-1を行っているため
    - outputsは単語IDを変換したベクトルになっているため，教師データはtext_generator.encode()を使って単語IDをベクトルに変換して誤差を計算

    ```python
    for i in range(seq_len - 1):
        loss += criterion(outputs[i], text_generator.encode(targets[i]))
    ```


<font color="blue">　【TASK】</font>誤差の計算（全ての時刻に教師データがある場合）を行いましょう  
  ループの設定は下記の通りです  
  - データ長seq_len - 1
  - rangeを使ってseq_len - 1と同じ要素数の配列を生成して，ループさせましょう
  

  誤差計算で与える引数は下記の通りです
  - ニューラルネットワークからの出力ouputs（i番目の要素のみ）
  - text_generatorクラスのencode()で変換した教師データtargets（i番目の要素のみ）
  

#### 4. 誤差逆伝播とパラメータの更新  

- パラメータの微分値を初期化
- 誤差逆伝播
- パラメータの更新
```python
# 【TASK】パラメータの微分値を初期化
# 【TASK】誤差逆伝播
# 【TASK】パラメータの更新
```

<font color="blue">　【TASK】</font>誤差逆伝播とパラメータの更新を行いましょう
- zero_grad()を使ってパラメータの微分値の初期化をしましょう  
  - [zero_grad()](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html#torch.optim.Optimizer.zero_grad)は最適化器が持っているので次のようにして呼び出します
  ```python
  optimizer_text_generator.zero_grad()
  ```

- backward()を使って誤差逆伝播させましょう
  - [backward()](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html)は計算結果を持った変数から次のようにして呼び出します
  ```python
  loss.backward()
  ```

- step()を使ってパラメータの更新を行いましょう
  - パラメータの更新は最適化器の持つ[step()](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html#torch.optim.Optimizer.step)は最適化器が持っているので次のようにして呼び出します
  ```python
  optimizer_text_generator.step()
  ```





#### 例題5のコード：後半の予測部分

In [ ]:
# 5. 文章の生成

# 最初に入力する単語のIDを取得
index = index2word_list.index("珈琲")
# Tensor型に変換
inputs = torch.tensor([[index]])

# 最初に入力するIDを単語に直して表示
print(index2word_list[index])

# 生成のループ
for i in range(seq_len):

    # ニューラルネットワークへのデータの入力
    inputs = inputs.to(device)
    outputs = text_generator(inputs)

    # ニューラルネットワークの出力をnumpy.ndarray型に変換
    outputs = outputs.to("cpu").detach().numpy()
    
    # Word2Vecを使って変換
    outputs = wv.similar_by_vector(outputs[0][0], topn=1)[0][0]

    # 単語のIDを取得
    index = index2word_list.index(outputs)

    # Tensor型に変換して次時刻の入力にする
    inputs = torch.tensor([[index]])

    # 予測単語の表示
    print(outputs)
    
    # 予測単語が。の場合にループを終える
    if index2word_list[index] == "美味しい":
        break

#### 5. 文章の生成

  - 文章の最初の単語をニューラルネットワークへ入力し，for文でループさせながら文章を生成する


- データの準備
  - indexという名前で単語IDを格納する変数を宣言
    - 単語IDの取得はlist形式配列のメソッドであるindex()を使う
    - 文章の最初の単語をindex()に与える
  - inputsという名前でTensor型に変換した単語IDを格納する変数を宣言

  ```python
  index = index2word_list.index("珈琲")
  inputs = torch.tensor([[index]])
  ```


- **<font color="red">【NEW!】</font>**for文でループさせながら以下の手順で文章を生成する
  - ニューラルネットワークへのデータの入力
    - ニューラルネットワークへデータを入力し出力を得る
  - ニューラルネットワークの出力をnumpy.ndarray型（以降，ndarray型）型に変換
  - Word2Vecを使って変換
    - 重みの値から単語を予測するにはsimilar_by_vector()を使う
    ```python
    outputs = wv.similar_by_vector(outputs[0][0], topn=1)[0][0]
    # 第１引数：単語の予測に使う重み
    # 第２引数：出力する候補の数
    # 戻り値1：最も値が近いベクトルのキー(list)
    ```

  - 単語のIDを取得
    - 単語IDの取得はlist形式配列のメソッドであるindex()を使う
    ```python
    index = index2word_list.index(outputs)
    # 第1引数：単語IDが欲しい単語
    # 戻り値1：該当するインデックス(int)
    ```

  - Tensor型に変換して次時刻の入力にする
  ```python
  inputs = torch.tensor([[index]])
  ```
  - ループ中に予測単語が`美味しい`になった場合はループを終える
  ```python
    if index2word_list[index]=="美味しい":
        break
  ```


<font color="blue">【TASK】</font>文章を生成してみましょう．`美味しい`がきたら予測のループをストップさせるので，うまく学習できていれば次のように表示されます．
```
珈琲
牛乳
が
美味しい
```

